# World Development Indicators (WDI): Descriptive Statistical Analysis
### Comparative macroeconomic profile of six economies | 2000–2023

**Purpose:** a descriptive, exploratory analysis of 12 major macroeconomic indicators. This notebook does not perform PCA, regression, hypothesis testing or causal inference. It describes levels, distributions, variation, country differences, time patterns and pairwise associations.

**Countries:** India, China, USA, Brazil, Germany and South Africa.  
**Period:** 2000–2023 (adjustable in Configuration).  
**Primary rule:** descriptive statistics are calculated from observed WDI values. Interpolated values are not silently substituted into the main analysis.

## 1. Setup and configuration
Edit `DATA_FOLDER` to the directory containing the World Bank `WDICSV` file. The optional `WDICountry` and `WDISeries` files enrich labels and metadata. In Google Colab, upload the CSV files into the session or mount Drive and set the folder path.

In [ ]:
%pip -q install plotly scipy openpyxl

In [ ]:
import os, glob, warnings, math
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import skew, kurtosis
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

# ---------------- CONFIGURATION ----------------
DATA_FOLDER = r"C:\Users\rbgir\OneDrive\Desktop\Ripu\Micro & Macro\Micro_macro_grp_assignment\WDI_CSV"
COUNTRIES = ["IND", "CHN", "USA", "BRA", "DEU", "ZAF"]
COUNTRY_LABELS = {
    "IND":"India", "CHN":"China", "USA":"United States",
    "BRA":"Brazil", "DEU":"Germany", "ZAF":"South Africa"
}
START_YEAR, END_YEAR = 2000, 2023

# 12 major macroeconomic factors drawn from the original notebook's WDI candidate list.
INDICATORS = {
    "GDP growth": "NY.GDP.MKTP.KD.ZG",                 # annual %
    "Inflation": "FP.CPI.TOTL.ZG",                     # annual %
    "Unemployment": "SL.UEM.TOTL.ZS",                  # % of labour force
    "GDP per capita": "NY.GDP.PCAP.CD",                # current US$
    "Trade (% GDP)": "NE.TRD.GNFS.ZS",                 # exports + imports / GDP
    "FDI net inflows (% GDP)": "BX.KLT.DINV.WD.GD.ZS", # % GDP
    "Government consumption (% GDP)": "NE.CON.GOVT.ZS",# % GDP
    "Exports (% GDP)": "NE.EXP.GNFS.ZS",               # % GDP
    "Imports (% GDP)": "NE.IMP.GNFS.ZS",               # % GDP
    "Gross capital formation (% GDP)": "NE.GDI.TOTL.ZS",# % GDP
    "Gross domestic savings (% GDP)": "NY.GDS.TOTL.ZS",# % GDP
    "Domestic credit to private sector (% GDP)": "FS.AST.PRVT.GD.ZS" # % GDP
}
# ------------------------------------------------

## 2. Load WDI files and verify the requested indicators

The main World Bank download is `WDICSV`. `WDICountry` and `WDISeries` are optional supporting files. The checks below stop early if the main data file or required country/indicator fields are missing.

In [ ]:
def find_file(folder, keyword):
    hits = [f for f in glob.glob(os.path.join(folder, "*"))
            if keyword.lower() in os.path.basename(f).lower()
            and f.lower().endswith((".csv", ".xlsx", ".xls"))]
    return sorted(hits, key=len)[0] if hits else None

def read_any(path):
    return pd.read_excel(path) if path.lower().endswith((".xlsx", ".xls")) else pd.read_csv(path, low_memory=False)

if not os.path.isdir(DATA_FOLDER):
    raise FileNotFoundError(
        f"DATA_FOLDER not found: {DATA_FOLDER}\n"
        "Edit DATA_FOLDER in the configuration cell to your WDI CSV folder."
    )

data_path = find_file(DATA_FOLDER, "WDICSV")
country_path = find_file(DATA_FOLDER, "WDICountry")
series_path = find_file(DATA_FOLDER, "WDISeries")
if data_path is None:
    raise FileNotFoundError("Could not find a file with 'WDICSV' in its name.")

raw = read_any(data_path)
country_meta = read_any(country_path) if country_path else None
series_meta = read_any(series_path) if series_path else None
raw.columns = [str(c).strip() for c in raw.columns]
year_cols = [c for c in raw.columns if c.isdigit()]
years = [str(y) for y in range(START_YEAR, END_YEAR + 1) if str(y) in year_cols]

required = {"Country Name", "Country Code", "Indicator Name", "Indicator Code"}
missing_cols = required - set(raw.columns)
if missing_cols:
    raise ValueError(f"WDICSV is missing required columns: {sorted(missing_cols)}")
if not years:
    raise ValueError("No year columns found for the selected analysis period.")

print(f"Loaded: {os.path.basename(data_path)} | rows={len(raw):,} | years={years[0]}–{years[-1]}")
print(f"Optional metadata: country={country_meta is not None}, series={series_meta is not None}")

## 3. Build the country-year analysis panel

The source is reshaped into one row per country-year. Missing values remain missing. The notebook does **not** interpolate the primary descriptive dataset because interpolation can artificially smooth volatility and change means, standard deviations and distribution shapes.

In [ ]:
selected_codes = list(INDICATORS.values())
found_ind = set(raw["Indicator Code"].astype(str))
found_cty = set(raw["Country Code"].astype(str))

indicator_audit = pd.DataFrame({
    "Indicator": list(INDICATORS.keys()),
    "WDI code": selected_codes,
    "Available in file": [x in found_ind for x in selected_codes]
})
country_audit = pd.DataFrame({
    "Country": [COUNTRY_LABELS.get(c, c) for c in COUNTRIES],
    "ISO3": COUNTRIES,
    "Available in file": [c in found_cty for c in COUNTRIES]
})
display(indicator_audit)
display(country_audit)

valid_codes = [c for c in COUNTRIES if c in found_cty]
valid_indicators = {k:v for k,v in INDICATORS.items() if v in found_ind}
if not valid_codes:
    raise ValueError("None of the configured country codes were found in WDICSV.")
if not valid_indicators:
    raise ValueError("None of the configured WDI indicator codes were found in WDICSV.")

sub = raw[
    raw["Country Code"].isin(valid_codes) &
    raw["Indicator Code"].isin(valid_indicators.values())
].copy()
long = sub.melt(
    id_vars=["Country Name","Country Code","Indicator Name","Indicator Code"],
    value_vars=years, var_name="year", value_name="value"
)
long["year"] = long["year"].astype(int)
long["value"] = pd.to_numeric(long["value"], errors="coerce")
code_to_name = {v:k for k,v in valid_indicators.items()}
long["indicator"] = long["Indicator Code"].map(code_to_name)
panel = (long.pivot_table(
    index=["Country Code","Country Name","year"],
    columns="indicator", values="value", aggfunc="first"
).reset_index())
panel.columns.name = None
panel["Country"] = panel["Country Code"].map(COUNTRY_LABELS).fillna(panel["Country Name"])
panel = panel.sort_values(["Country Code","year"]).reset_index(drop=True)

print(f"Panel: {panel.shape[0]:,} country-year rows × {len(valid_indicators)} indicators")
display(panel.head())

## 4. Data quality and coverage audit

Coverage is reported at both indicator and country-indicator levels. Low coverage is a substantive limitation, not a reason to manufacture values. Missingness can vary by indicator and country.

In [ ]:
indicator_order = list(valid_indicators.keys())
coverage = pd.DataFrame({
    "Indicator": indicator_order,
    "Observed values": [panel[v].notna().sum() for v in indicator_order],
    "Possible values": [len(panel) for v in indicator_order],
    "Coverage (%)": [100*panel[v].notna().mean() for v in indicator_order]
}).sort_values("Coverage (%)", ascending=True)
coverage["Missing values"] = coverage["Possible values"] - coverage["Observed values"]
display(coverage.round(1))

country_coverage = (panel.groupby("Country")[indicator_order]
                    .agg(lambda s: 100*s.notna().mean()).round(1))
display(country_coverage)

fig = px.bar(coverage, x="Coverage (%)", y="Indicator", orientation="h",
             color="Coverage (%)", color_continuous_scale="Tealgrn",
             text=coverage["Coverage (%)"].map(lambda x:f"{x:.1f}%"),
             title="Chart 1. Indicator data coverage across selected country-years",
             template="plotly_white")
fig.update_layout(height=650, coloraxis_showscale=False, xaxis_title="Observed values (%)")
fig.show()

## 5. Descriptive statistics: central tendency, dispersion and shape

For each indicator, the notebook reports **N, mean, median, standard deviation, variance, minimum, Q1, Q3, maximum, range, IQR, coefficient of variation, skewness and excess kurtosis**. Median and IQR are useful when distributions are skewed or contain extreme observations. CV is shown only as a relative-dispersion aid and should not be compared mechanically across variables with zero/negative means.

In [ ]:
def descriptive_table(df, variables):
    rows=[]
    for v in variables:
        s=df[v].dropna()
        if len(s)==0:
            continue
        mean=s.mean()
        sd=s.std(ddof=1)
        rows.append({
            "Indicator":v, "N":len(s), "Mean":mean, "Median":s.median(),
            "Std. deviation":sd, "Variance":s.var(ddof=1),
            "Min":s.min(), "Q1":s.quantile(.25), "Q3":s.quantile(.75),
            "Max":s.max(), "Range":s.max()-s.min(),
            "IQR":s.quantile(.75)-s.quantile(.25),
            "CV (%)":(sd/abs(mean)*100) if pd.notna(mean) and mean!=0 else np.nan,
            "Skewness":s.skew(), "Excess kurtosis":s.kurt()
        })
    return pd.DataFrame(rows).set_index("Indicator")

desc = descriptive_table(panel, indicator_order)
display(desc.round(3))

## 6. Country-level descriptive profile

The table below compares country means and medians. These are period averages across available observations, not a ranking of overall welfare or performance. For variables with gaps, each country's effective sample size differs.

In [ ]:
country_stats = panel.groupby("Country")[indicator_order].agg(["count","mean","median","std"])
display(country_stats.round(2))

## 7. Modern chart gallery: 15 descriptive views

Each chart answers a distinct descriptive question. Interpretations are generated from the observed data at runtime. Correlations and scatterplots show association only; they do not establish causality.

In [ ]:
# Chart 2 — coverage matrix by country and indicator
cov_matrix = panel.groupby("Country")[indicator_order].agg(lambda s: 100*s.notna().mean())
fig = px.imshow(cov_matrix, text_auto=".0f", aspect="auto", color_continuous_scale="YlGnBu",
                title="Chart 2. Data coverage heatmap by country and indicator",
                labels=dict(x="Indicator", y="Country", color="Coverage (%)"), template="plotly_white")
fig.update_layout(height=520)
fig.show()

# Chart 3 — GDP growth trend
if "GDP growth" in panel:
    fig=px.line(panel, x="year", y="GDP growth", color="Country", markers=True,
                title="Chart 3. GDP growth over time", template="plotly_white")
    fig.add_hline(y=0, line_dash="dash", line_color="gray")
    fig.update_layout(hovermode="x unified", yaxis_title="GDP growth (annual %)")
    fig.show()

# Chart 4 — Inflation trend
if "Inflation" in panel:
    fig=px.line(panel, x="year", y="Inflation", color="Country", markers=True,
                title="Chart 4. Consumer price inflation over time", template="plotly_white")
    fig.add_hline(y=0, line_dash="dash", line_color="gray")
    fig.update_layout(hovermode="x unified", yaxis_title="Inflation (annual %)")
    fig.show()

# Chart 5 — Unemployment trend
if "Unemployment" in panel:
    fig=px.line(panel, x="year", y="Unemployment", color="Country", markers=True,
                title="Chart 5. Unemployment rate over time", template="plotly_white")
    fig.update_layout(hovermode="x unified", yaxis_title="% of labour force")
    fig.show()

# Chart 6 — GDP per capita indexed to first available year (=100)
if "GDP per capita" in panel:
    temp=panel[["Country","year","GDP per capita"]].dropna().copy()
    temp["Index (first observed year=100)"]=temp.groupby("Country")["GDP per capita"].transform(
        lambda s: 100*s/s.iloc[0] if len(s) else s)
    fig=px.line(temp, x="year", y="Index (first observed year=100)", color="Country", markers=True,
                title="Chart 6. GDP per capita growth index (first observed year = 100)",
                template="plotly_white")
    fig.show()

# Chart 7 — GDP growth distribution by country (violin)
if "GDP growth" in panel:
    fig=px.violin(panel.dropna(subset=["GDP growth"]), x="Country", y="GDP growth",
                  color="Country", box=True, points="outliers",
                  title="Chart 7. GDP growth distribution and country-level spread",
                  template="plotly_white")
    fig.add_hline(y=0, line_dash="dash", line_color="gray")
    fig.update_layout(showlegend=False)
    fig.show()

# Chart 8 — Inflation distribution (histogram + density)
if "Inflation" in panel:
    fig=px.histogram(panel.dropna(subset=["Inflation"]), x="Inflation", color="Country",
                     marginal="violin", opacity=.72, barmode="overlay", nbins=35,
                     title="Chart 8. Inflation distribution across all observed country-years",
                     template="plotly_white")
    fig.show()

# Chart 9 — Box plots for all percentage-style indicators (exclude GDP per capita)
pct_vars=[v for v in indicator_order if v!="GDP per capita"]
if pct_vars:
    boxdata=panel.melt(id_vars=["Country","year"], value_vars=pct_vars,
                       var_name="Indicator", value_name="Value").dropna()
    # Standardise within indicator for visual comparison of spread only
    boxdata["Within-indicator z-score"]=boxdata.groupby("Indicator")["Value"].transform(
        lambda s:(s-s.mean())/s.std(ddof=0) if s.std(ddof=0)>0 else 0)
    fig=px.box(boxdata, x="Indicator", y="Within-indicator z-score", color="Indicator",
               points="outliers", title="Chart 9. Relative dispersion across macro indicators (z-scores)",
               template="plotly_white")
    fig.update_layout(showlegend=False, xaxis_tickangle=-35)
    fig.show()

# Chart 10 — Correlation heatmap (Pearson)
corr=panel[indicator_order].corr(method="pearson", min_periods=8)
fig=px.imshow(corr, text_auto=".2f", zmin=-1, zmax=1, color_continuous_scale="RdBu_r",
              aspect="auto", title="Chart 10. Pearson correlation matrix (pairwise available observations)",
              template="plotly_white")
fig.update_layout(height=720)
fig.show()

# Chart 11 — GDP growth vs gross capital formation
if {"GDP growth","Gross capital formation (% GDP)"}.issubset(panel.columns):
    fig=px.scatter(panel, x="Gross capital formation (% GDP)", y="GDP growth",
                   color="Country", size="GDP per capita" if "GDP per capita" in panel else None,
                   hover_name="Country", hover_data=["year"], opacity=.75,
                   title="Chart 11. GDP growth and gross capital formation",
                   template="plotly_white")
    fig.show()

# Chart 12 — Trade openness vs GDP growth
if {"Trade (% GDP)","GDP growth"}.issubset(panel.columns):
    fig=px.scatter(panel, x="Trade (% GDP)", y="GDP growth", color="Country",
                   hover_data=["year"], opacity=.75,
                   title="Chart 12. Trade openness and GDP growth",
                   template="plotly_white")
    fig.show()

# Chart 13 — FDI distribution by country
if "FDI net inflows (% GDP)" in panel:
    fig=px.box(panel, x="Country", y="FDI net inflows (% GDP)", color="Country",
               points="all", title="Chart 13. Foreign direct investment inflows (% GDP)",
               template="plotly_white")
    fig.update_layout(showlegend=False)
    fig.show()

# Chart 14 — Latest available year country comparison (heatmap of within-indicator z-scores)
latest_rows=[]
for c,g in panel.sort_values("year").groupby("Country"):
    row={"Country":c}
    for v in indicator_order:
        z=g.loc[g[v].notna(), ["year",v]]
        if len(z):
            row[v]=z.iloc[-1][v]
            row[v+"_year"]=int(z.iloc[-1]["year"])
    latest_rows.append(row)
latest=pd.DataFrame(latest_rows).set_index("Country")
latest_values=latest[indicator_order].copy()
latest_z=(latest_values-latest_values.mean())/latest_values.std(ddof=0)
fig=px.imshow(latest_z, text_auto=".1f", aspect="auto", color_continuous_scale="RdBu_r",
              title="Chart 14. Latest available country profile (standardised within each indicator)",
              labels=dict(x="Indicator", y="Country", color="Z-score"), template="plotly_white")
fig.update_layout(height=480)
fig.show()

# Chart 15 — Year-by-year median GDP growth heatmap
if "GDP growth" in panel:
    annual=panel.pivot_table(index="Country", columns="year", values="GDP growth", aggfunc="median")
    fig=px.imshow(annual, text_auto=".1f", aspect="auto", color_continuous_scale="RdYlBu_r",
                  title="Chart 15. GDP growth heatmap: country × year",
                  labels=dict(x="Year", y="Country", color="GDP growth (%)"), template="plotly_white")
    fig.update_layout(height=430)
    fig.show()

## 8. Automated descriptive interpretation

These are data-led descriptive statements, not causal conclusions. The code reports the highest/lowest period means, most volatile indicators, distribution shape and strongest pairwise correlations, while explicitly showing the sample sizes used.

In [ ]:
def safe_num(x, digits=2):
    return "NA" if pd.isna(x) else f"{x:,.{digits}f}"

# Coverage interpretation
best_cov=coverage.sort_values("Coverage (%)",ascending=False).iloc[0]
worst_cov=coverage.sort_values("Coverage (%)",ascending=True).iloc[0]
display(Markdown(
    f"### Data coverage\n"
    f"- Highest coverage: **{best_cov['Indicator']}** ({best_cov['Coverage (%)']:.1f}%).\n"
    f"- Lowest coverage: **{worst_cov['Indicator']}** ({worst_cov['Coverage (%)']:.1f}%). "
    f"Interpret results for low-coverage indicators cautiously."
))

# Indicator variability
vol=desc.sort_values("Std. deviation",ascending=False)
display(Markdown("### Dispersion\n" +
    f"- Largest raw standard deviation: **{vol.index[0]}** (SD = {safe_num(vol.iloc[0]['Std. deviation'])}). " +
    "Because indicators use different units, raw SDs should not be compared as a measure of relative volatility."))
cv_rank=desc["CV (%)"].replace([np.inf,-np.inf],np.nan).dropna().sort_values(ascending=False)
if len(cv_rank):
    display(Markdown(f"- Highest coefficient of variation among defined values: **{cv_rank.index[0]}** ({cv_rank.iloc[0]:.1f}%). CV can be unstable when the mean is close to zero."))

# GDP growth country ranking
if "GDP growth" in panel:
    gmean=panel.groupby("Country")["GDP growth"].mean().dropna().sort_values(ascending=False)
    if len(gmean):
        display(Markdown(f"### GDP growth\n- Highest average observed GDP growth: **{gmean.index[0]}** ({gmean.iloc[0]:.2f}%).\n- Lowest average observed GDP growth: **{gmean.index[-1]}** ({gmean.iloc[-1]:.2f}%)."))

# Distribution shape
shape=desc[["Skewness","Excess kurtosis"]].dropna()
if len(shape):
    most_skew=shape["Skewness"].abs().idxmax()
    display(Markdown(f"### Distribution shape\n- Most asymmetric distribution by absolute skewness: **{most_skew}** (skewness = {shape.loc[most_skew,'Skewness']:.2f}). A positive value indicates a longer right tail; a negative value indicates a longer left tail."))

# Correlations with pairwise N
if corr.shape[0] >= 2:
    mask=np.triu(np.ones(corr.shape,dtype=bool),k=1)
    pairs=corr.where(mask).stack().rename("Pearson r").reset_index()
    pairs.columns=["Indicator 1","Indicator 2","Pearson r"]
    pairs["Absolute r"]=pairs["Pearson r"].abs()
    pair_n=panel[indicator_order].notna().astype(int).T @ panel[indicator_order].notna().astype(int)
    pairs["Pairwise N"]=[int(pair_n.loc[a,b]) for a,b in zip(pairs["Indicator 1"],pairs["Indicator 2"])]
    display(pairs.sort_values("Absolute r",ascending=False).head(10).drop(columns="Absolute r").round(3))
    if len(pairs):
        top=pairs.sort_values("Absolute r",ascending=False).iloc[0]
        display(Markdown(f"- Strongest absolute pairwise Pearson correlation: **{top['Indicator 1']}** and **{top['Indicator 2']}** (r = {top['Pearson r']:.2f}; pairwise N = {int(top['Pairwise N'])}). This is an association, not evidence of causation."))

## 9. Methodological notes and limitations

1. **Descriptive, not causal:** trends, differences and correlations do not establish that one macroeconomic factor causes another.
2. **Unequal coverage:** WDI indicators can have different missingness patterns. Always read `N` and coverage alongside each statistic or chart.
3. **No silent imputation:** main tables and charts use observed values only. This preserves the observed distribution but means sample sizes can vary.
4. **Current-dollar GDP per capita:** this indicator is affected by inflation and exchange-rate movements. It is not a constant-price measure of real income. If the assignment requires real comparisons, consider replacing it with `NY.GDP.PCAP.KD` (constant 2015 US$).
5. **Mixed units:** GDP per capita is in US dollars, while most other indicators are percentages. Raw standard deviations across units are not directly comparable.
6. **Correlation:** Pearson correlation captures linear association and can be influenced by outliers. A Spearman correlation can be added as a rank-based robustness description.
7. **Small country set:** six economies provide a focused comparative case study, not a representative sample of all countries.
8. **WDI revisions:** results can differ across download dates because the World Bank revises historical series.

### Suggested reporting structure
- Dataset and country/year scope
- Indicator definitions and coverage
- Descriptive-statistics table
- Country comparison and time trends
- Distribution and volatility findings
- Correlation patterns
- Limitations and concise conclusion

## 10. Export descriptive tables

The following cell exports the descriptive statistics, coverage and country summary tables to an Excel workbook for use in your report or presentation.

In [ ]:
output_file = os.path.join(DATA_FOLDER, "WDI_Descriptive_Statistics_Tables.xlsx")
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    desc.to_excel(writer, sheet_name="Overall Descriptives")
    coverage.to_excel(writer, sheet_name="Coverage", index=False)
    country_stats.to_excel(writer, sheet_name="Country Statistics")
    country_coverage.to_excel(writer, sheet_name="Country Coverage")
    latest.to_excel(writer, sheet_name="Latest Values")
print("Excel tables exported to:", output_file)